# Naive Bayes Churn Prediction Model
## HPB Fintech Hackathon 2026

### Pipeline
1. Load cleaned features from data preparation
2. Train/test split (stratified, 75/25)
3. Feature scaling (StandardScaler)
4. Train Gaussian Naive Bayes
5. Optimize classification threshold for **F1 score**
6. Evaluate model performance
7. Generate per-customer churn risk scores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

DATA = Path('../data')
df = pd.read_csv(DATA / 'churn_features_clean.csv')

# Identify feature columns (everything except ID, period, churned)
META_COLS = ['IDENTIFIKATOR_KLIJENTA', 'period', 'churned']
FEATURES = [c for c in df.columns if c not in META_COLS]
TARGET = 'churned'

X = df[FEATURES]
y = df[TARGET]

print(f'Dataset: {len(df):,} samples, {len(FEATURES)} features')
print(f'Churn rate: {y.mean():.1%}')
print(f'Features: {FEATURES}')
X.describe().round(2)

In [ ]:
# ── Train/Test Split & Scaling ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train churn: {y_train.mean():.1%}  |  Test churn: {y_test.mean():.1%}')

# ── Train Gaussian NB ──
gnb = GaussianNB()
gnb.fit(X_train_s, y_train)

y_proba = gnb.predict_proba(X_test_s)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f'\nGaussianNB trained — ROC AUC: {auc:.4f}')

In [ ]:
# ── Threshold Optimization (maximize F1) ──
thresholds = np.arange(0.01, 1.0, 0.01)
f1s, precs, recs = [], [], []
for t in thresholds:
    yp = (y_proba >= t).astype(int)
    f1s.append(f1_score(y_test, yp, zero_division=0))
    precs.append(precision_score(y_test, yp, zero_division=0))
    recs.append(recall_score(y_test, yp, zero_division=0))

best_idx = np.argmax(f1s)
best_t = thresholds[best_idx]
best_f1 = f1s[best_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, f1s, color='#2c3e50', lw=2.5, label='F1 Score')
ax.plot(thresholds, precs, color='#3498db', lw=1.5, ls='--', label='Precision')
ax.plot(thresholds, recs, color='#e74c3c', lw=1.5, ls='--', label='Recall')
ax.axvline(best_t, color='#27ae60', lw=2, ls=':', label=f'Best threshold = {best_t:.2f}')
ax.scatter([best_t], [best_f1], color='#27ae60', s=120, zorder=5, edgecolors='black')
ax.annotate(f'F1 = {best_f1:.4f}', xy=(best_t, best_f1),
            xytext=(best_t + 0.08, best_f1 - 0.06), fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round,pad=0.3', fc='#eafaf1', ec='#27ae60'))
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Optimization — Maximizing F1 Score', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'Optimal threshold: {best_t:.2f}')
print(f'F1: {best_f1:.4f}  |  Precision: {precs[best_idx]:.4f}  |  Recall: {recs[best_idx]:.4f}')

In [ ]:
# ── Model Evaluation at Optimal Threshold ──
y_pred = (y_proba >= best_t).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (threshold={best_t:.2f})', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#2c3e50', lw=2.5, label=f'GaussianNB (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', ls='--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# ── Risk Scores for All Customers ──
X_all_s = scaler.transform(X)
risk_proba = gnb.predict_proba(X_all_s)[:, 1]

risk = df[['IDENTIFIKATOR_KLIJENTA', 'period']].copy()
risk['churn_risk_score'] = risk_proba
risk['predicted_churn'] = (risk_proba >= best_t).astype(int)
risk = risk.sort_values('churn_risk_score', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(risk['churn_risk_score'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(best_t, color='#e74c3c', lw=2, ls='--', label=f'Threshold = {best_t:.2f}')
axes[0].set_xlabel('Churn Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution', fontweight='bold')
axes[0].legend()

# By actual churn status
for val, label, color in [(0, 'Active', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    subset = risk_proba[y == val]
    axes[1].hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
axes[1].axvline(best_t, color='black', lw=2, ls='--', label=f'Threshold = {best_t:.2f}')
axes[1].set_xlabel('Churn Risk Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Score by Actual Churn Status', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Save
risk_path = DATA / 'churn_risk_scores.csv'
risk.to_csv(risk_path, index=False)

n_high = (risk['predicted_churn'] == 1).sum()
n_low = len(risk) - n_high
print(f'Saved: {risk_path}')
print(f'  High risk: {n_high:,} ({n_high/len(risk):.1%})')
print(f'  Low risk:  {n_low:,} ({n_low/len(risk):.1%})')
print(f'\nTop 10 highest-risk customers:')
risk.head(10)